In [1]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

In [2]:
pokemon = pd.read_csv("data/raw_data/pokemon.csv")
combats = pd.read_csv("data/raw_data/combats.csv")
pokemonfullstats = pd.read_csv("data/raw_data/pokemonfullstats.csv")

## **1. Missing Values**

In [3]:
missing_values = pokemon.isna().mean()
missing_ratio = missing_values[missing_values > 0].sort_values(ascending=False).to_frame("missing_ratio")
display(missing_ratio.style.background_gradient(cmap="Reds").format("{:.1%}"))

,missing_ratio
Type 2,48.2%
Name,0.1%


In [4]:
missing_values = pokemonfullstats.isna().mean()
missing_ratio = missing_values[missing_values > 0].sort_values(ascending=False).to_frame("missing_ratio")
display(missing_ratio.style.background_gradient(cmap="Reds").format("{:.1%}"))

,missing_ratio


In [5]:
missing_values = combats.isna().mean()
missing_ratio = missing_values[missing_values > 0].sort_values(ascending=False).to_frame("missing_ratio")
display(missing_ratio.style.background_gradient(cmap="Reds").format("{:.1%}"))

,missing_ratio


In [6]:
na_pokemon = pokemon.isna().sum()
na_pokemonfs = pokemonfullstats.isna().sum()
na_combats = combats.isna().sum()
print(f'Missing value in pokemon: {sum(na_pokemon[na_pokemon > 0])}')
print(f'Missing value in pokemonfullstats: {sum(na_pokemonfs[na_pokemonfs > 0])}')
print(f'Missing value in combats: {sum(na_combats[na_combats > 0])}')

Missing value in pokemon: 387
Missing value in pokemonfullstats: 0
Missing value in combats: 0


Ta thấy `pokemon.csv` có 1 missing value ở cột Name và rất nhiều missing values ở cột `Type 2` do nhiều Pokémon chỉ có một thuộc tính chính

Ở cột `Type 2`, missing value có thể được giải quyết đơn giản bằng cách thêm giá trị 'None' để biểu thị cho việc Pokémon đó chỉ có một thuộc tính chính

In [7]:
pokemon.fillna({'Type 2': 'None'}, inplace=True)

Còn ở cột Name, giả dụ tên của các Pokémon thường sẽ được xếp theo thứ tự ID nên hãy xem qua các dòng xung quanh của missing value này

In [8]:
missing_point = pokemon.index[pokemon['Name'].isna()][0]
pokemon.loc[missing_point - 3:missing_point + 3, ['Name', 'Type 1', 'Type 2']]

,Name,Type 1,Type 2
59,Psyduck,Water,None
60,Golduck,Water,None
61,Mankey,Fighting,None
62,NaN,Fighting,None
63,Growlithe,Fire,None
64,Arcanine,Fire,None
65,Poliwag,Water,None


Giá trị bị thiếu nằm giữa hai Pokémon có tên 'Mankey' và 'Growlithe', có thể suy luận giá trị từ các web tổng hợp thông tin pokemon,

hoặc xem từ pokemonfullstats.csv vì dataset này không có giá trị thiếu trên cột Name

In [9]:
target_point = pokemonfullstats.index[pokemonfullstats['Name'] == 'Mankey'][0]
pokemonfullstats.loc[target_point - 2:target_point + 4, ['Name', 'Type']]

,Name,Type
65,Psyduck,['Water']
66,Golduck,['Water']
67,Mankey,['Fighting']
68,Primeape,['Fighting']
69,Growlithe,['Fire']
70,Hisuian Growlithe,"['Fire', 'Rock']"
71,Arcanine,['Fire']


Có thể thấy giá trị bị thiếu chính là `Primeape`

## **2. Duplicating Values**

In [10]:
dup_pokemon = pokemon.duplicated().sum()
dup_pokemonfs = pokemonfullstats.duplicated().sum()
dup_combats = combats.duplicated().sum()
print(f'Duplicated value in pokemon: {sum(dup_pokemon[dup_pokemon > 0])}')
print(f'Duplicated value in pokemonfullstats: {sum(dup_pokemonfs[dup_pokemonfs > 0])}')
print(f'Duplicated value in combats: {sum(dup_combats[dup_combats > 0])}')

Duplicated value in pokemon: 0
Duplicated value in pokemonfullstats: 0
Duplicated value in combats: 1952


In [11]:
dup_combats_indices = combats[combats.duplicated()].index
combats = combats.drop(dup_combats_indices).reset_index(drop=True)

## **3. Special Issues of the Dataset**

In [12]:
pokemonfullstats.drop(columns=['DexNumber'], inplace=True)

Để trực quan hoá một cách hiệu quả bộ dữ liệu về Pokémon, trước hết cần kết hợp thông tin từ cả hai bộ dữ liệu pokemon.csv và pokemonfullstats.csv để có được bảng dữ liệu đầy

đủ về đặc tính của các Pokémon Nhưng có một số vấn đề cần phải giải quyết trước khi thực hiện việc này:

Dễ thấy cách đánh số ID của các Pokémon trong hai bộ dữ liệu không đồng nhất, ví dụ như trong pokemon.csv, Charmander có ID ở cột # là 5, nhưng trong pokemonfullstats.csv,

lại có DexNumber là 4, và sự chênh lệch này sẽ lặp lại đến cuối bộ dữ liệu

Nguyên nhân gây ra sự chênh lệch này là do trong pokemonfullstats.csv, dữ liệu được lấy từ serebii.net và bulbapedia, và theo quy ước của các trang web này, các Pokémon tiến 

hoá ở bậc Mega sẽ nằm cùng một trang với Pokémon gốc của nó và có cùng số Pokedex, rất có thể thay đổi này được thực hiện sau khi bộ dữ liệu pokemon.csv được thu thập dẫn đến 

sự chênh lệch này

Tuy nhiên bộ dữ liệu pokemon.csv có cùng nguồn gốc với combats.csv, nên để đảm bảo tính nhất quán của dữ liệu, chúng em vẫn sẽ sử dụng cách đánh số ID từ pokemon.csv làm 

chuẩn, và merge hai bảng dữ liệu dựa trên cột Name thay vì cột # và DexNumber như thông thường, điều này đồng nghĩa cột DexNumber trong pokemonfullstats.csv sẽ không còn giá 

trị sử dụng

Tiếp đó, trong cả 2 bộ dữ liệu cũng có một số cột có nội dung tương tự nhau, ví dụ như các chỉ số chiến đấu (HP, Attack, Defense, Special Attack, Special Defense, Speed),

để tránh việc dư thừa thông tin khi kết hợp hai bảng dữ liệu, chúng em sẽ giữ lại các cột từ pokemonfullstats.csv vì bộ dữ liệu này có nhiều thông tin chi tiết hơn về các 

đặc điểm của Pokémon và loại bỏ các cột tương ứng từ pokemon.csv

In [13]:
pokemon.drop(columns=['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary'], inplace=True)

## **Merge `pokemon` with `pokemonfullstats` => `merged_pokemon`**

In [14]:
merged_pokemon = pd.merge(pokemon, pokemonfullstats, left_on='Name', right_on='Name', how='outer')
merged_pokemon = merged_pokemon.dropna(subset=['#'])
merged_pokemon.sort_values(by=['#'], inplace=True, ignore_index=True)
merged_pokemon.rename(columns={'#': 'ID'}, inplace=True)

In [15]:
merged_pokemon.head()

,ID,Name,Type 1,Type 2,Type,Abilities,HiddenAbility,Generation,Hp,Attack,...,DamageFromSteel,DamageFromFire,DamageFromWater,DamageFromGrass,DamageFromElectric,DamageFromPsychic,DamageFromIce,DamageFromDragon,DamageFromDark,DamageFromFairy
0,1.0,Bulbasaur,Grass,Poison,"['Grass', 'Poison']",['Overgrow'],['Chlorophyll'],I,45.0,49.0,...,1.0,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5
1,2.0,Ivysaur,Grass,Poison,"['Grass', 'Poison']",['Overgrow'],['Chlorophyll'],I,60.0,62.0,...,1.0,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5
2,3.0,Venusaur,Grass,Poison,"['Grass', 'Poison']",['Overgrow'],['Chlorophyll'],I,80.0,82.0,...,1.0,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5
3,4.0,Mega Venusaur,Grass,Poison,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,Charmander,Fire,None,['Fire'],['Blaze'],['Solar Power'],I,39.0,52.0,...,0.5,0.5,2.0,0.50,1.0,1.0,0.5,1.0,1.0,0.5


In [16]:
with open('data/modified/merged_pokemon.csv', 'w', encoding = 'utf-8') as f:
    merged_pokemon.to_csv(f, index=False)

## **Merge `merged_pokemon` with `combats` => file `dirty_data` để train model với dữ liệu trước khi xử lý**

In [17]:
dirtydata = pd.read_csv('data/modified/merged_pokemon.csv')

dirty_data = pd.merge(combats, dirtydata, left_on='First_pokemon', right_on='ID', how='inner')
dirty_data.drop(columns=['ID'], inplace=True)
dirty_data.rename(columns={c: f"{c}_P1" for c in dirty_data.columns[3:]}, inplace=True)

dirty_data = pd.merge(dirty_data, dirtydata, left_on='Second_pokemon', right_on='ID', how='inner')
dirty_data.drop(columns=['ID'], inplace=True)
dirty_data.rename(columns={c: f"{c}_P2" for c in dirty_data.columns[3 + dirtydata.shape[1] - 1:]}, inplace=True)

dirty_data.dropna(inplace=True)

dirty_data.to_csv('data/modified/dirty_model.csv', index=False)

In [18]:
df = pd.read_csv('data/modified/dirty_model.csv')
df.head()

,First_pokemon,Second_pokemon,Winner,Name_P1,Type 1_P1,Type 2_P1,Type_P1,Abilities_P1,HiddenAbility_P1,Generation_P1,...,DamageFromSteel_P2,DamageFromFire_P2,DamageFromWater_P2,DamageFromGrass_P2,DamageFromElectric_P2,DamageFromPsychic_P2,DamageFromIce_P2,DamageFromDragon_P2,DamageFromDark_P2,DamageFromFairy_P2
0,266,298,298,Larvitar,Rock,Ground,"['Rock', 'Ground']",['Guts'],['Sand Veil'],II,...,1.0,2.0,0.5,0.5,0.5,0.0,2.0,1.0,0.5,2.0
1,702,701,701,Virizion,Grass,Fighting,"['Grass', 'Fighting']",['Justified'],[],V,...,2.0,0.5,2.0,2.0,1.0,2.0,1.0,1.0,0.5,2.0
2,151,231,151,Omastar,Rock,Water,"['Rock', 'Water']","['Swift Swim', 'Shell Armor']",['Weak Armor'],I,...,2.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,657,752,657,Joltik,Bug,Electric,"['Bug', 'Electric']","['Compound Eyes', 'Unnerve']",['Swarm'],V,...,0.5,2.0,1.0,0.5,1.0,0.5,0.5,0.5,2.0,0.5
4,192,134,134,Natu,Psychic,Flying,"['Psychic', 'Flying']","['Synchronize', 'Early Bird']",['Magic Bounce'],II,...,2.0,2.0,1.0,1.0,1.0,0.5,0.5,1.0,2.0,1.0


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9661 entries, 0 to 9660
Data columns (total 99 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   First_pokemon          9661 non-null   int64  
 1   Second_pokemon         9661 non-null   int64  
 2   Winner                 9661 non-null   int64  
 3   Name_P1                9661 non-null   object 
 4   Type 1_P1              9661 non-null   object 
 5   Type 2_P1              9661 non-null   object 
 6   Type_P1                9661 non-null   object 
 7   Abilities_P1           9661 non-null   object 
 8   HiddenAbility_P1       9661 non-null   object 
 9   Generation_P1          9661 non-null   object 
 10  Hp_P1                  9661 non-null   float64
 11  Attack_P1              9661 non-null   float64
 12  Defense_P1             9661 non-null   float64
 13  SpecialAttack_P1       9661 non-null   float64
 14  SpecialDefense_P1      9661 non-null   float64
 15  Spee

In [20]:
df.duplicated().sum()

np.int64(0)

In [21]:
df['Winner'] = np.where(df['Winner'] == df['First_pokemon'], 0, 1)

In [22]:
obj_cols = df.select_dtypes(include=['object']).columns
obj_cols
df = df.drop(columns=obj_cols)
df.drop(columns = [ 'Speed_P1','Speed_P2', 'TotalStats_P1','TotalStats_P2', 'Hp_P1',
                   'Attack_P1', 'Defense_P1', 'SpecialAttack_P1', 'SpecialDefense_P1',
                   'SpecialAttack_P2', 'SpecialDefense_P2', 'CatchRate_P1', 'CatchRate_P2',
                   'EvoStage_P1','EggCycles_P1','Height_P1','Weight_P1','Hp_P1',
                   'DamageFromGround_P1','TotalEvoStages_P1','Hp_P2', 'Attack_P2','Weight_P2',
                   'Height_P2','DamageFromGrass_P1','Attack_P2', 'Weight_P2', 'Height_P2',
                   'Defense_P2', 'DamageFromGrass_P1', 'DamageFromFighting_P1', 'DamageFromIce_P1', 
                   'DamageFromBug_P1', 'DamageFromElectric_P1', 'DamageFromRock_P1', 
                   'DamageFromFire_P1', 'DamageFromFlying_P1', 'DamageFromWater_P1', 
                   'DamageFromSteel_P1', 'EvoStage_P2', 'DamageFromPoison_P1', 'EggCycles_P2', 
                   'DamageFromPsychic_P1', 'DamageFromGhost_P1', 'DamageFromFairy_P1', 
                   'TotalEvoStages_P2', 'DamageFromDark_P1', 'BaseFriendship_P1', 
                   'DamageFromNormal_P1', 'DamageFromGround_P2', 'DamageFromDragon_P1'],
                    inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9661 entries, 0 to 9660
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   First_pokemon          9661 non-null   int64  
 1   Second_pokemon         9661 non-null   int64  
 2   Winner                 9661 non-null   int64  
 3   IsLegendary_P1         9661 non-null   float64
 4   IsMythical_P1          9661 non-null   float64
 5   IsUltraBeast_P1        9661 non-null   float64
 6   HasMega_P1             9661 non-null   float64
 7   BaseFriendship_P2      9661 non-null   float64
 8   IsLegendary_P2         9661 non-null   float64
 9   IsMythical_P2          9661 non-null   float64
 10  IsUltraBeast_P2        9661 non-null   float64
 11  HasMega_P2             9661 non-null   float64
 12  DamageFromNormal_P2    9661 non-null   float64
 13  DamageFromFighting_P2  9661 non-null   float64
 14  DamageFromFlying_P2    9661 non-null   float64
 15  Dama

In [23]:
X = df.drop(columns=['First_pokemon', 'Second_pokemon', 'Winner'])
y = df['Winner']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

# **Random Forest**

In [24]:
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

TOP_N = 30
top_features_df = importances.sort_values(by="Importance", ascending=False).head(TOP_N)

selected_features = top_features_df["Feature"].tolist()

print(f"----- {TOP_N} Feature Importances -----")
print(top_features_df)

----- 30 Feature Importances -----
                  Feature  Importance
0          IsLegendary_P1    0.125902
3              HasMega_P1    0.081991
1           IsMythical_P1    0.059926
8              HasMega_P2    0.054709
4       BaseFriendship_P2    0.052062
19     DamageFromGrass_P2    0.049675
5          IsLegendary_P2    0.045280
20  DamageFromElectric_P2    0.037680
14       DamageFromBug_P2    0.036851
22       DamageFromIce_P2    0.036536
11    DamageFromFlying_P2    0.036492
13      DamageFromRock_P2    0.036259
25     DamageFromFairy_P2    0.035029
18     DamageFromWater_P2    0.034289
17      DamageFromFire_P2    0.033590
12    DamageFromPoison_P2    0.032999
10  DamageFromFighting_P2    0.032885
16     DamageFromSteel_P2    0.031979
15     DamageFromGhost_P2    0.031459
21   DamageFromPsychic_P2    0.026917
24      DamageFromDark_P2    0.026680
23    DamageFromDragon_P2    0.023691
9     DamageFromNormal_P2    0.022248
6           IsMythical_P2    0.014872
7         IsUlt

In [25]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predict & evaluate
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy (Random Forest): {accuracy:.4f}")
print(classification_report(y_test, y_pred))

Accuracy (Random Forest): 0.6891
              precision    recall  f1-score   support

           0       0.68      0.64      0.66       902
           1       0.70      0.73      0.72      1031

    accuracy                           0.69      1933
   macro avg       0.69      0.69      0.69      1933
weighted avg       0.69      0.69      0.69      1933



In [26]:
# Train Logistic Regression không feature selection
logreg = LogisticRegression(solver='lbfgs', max_iter=1000, n_jobs=-1)
logreg.fit(X_train, y_train)

# Predict & evaluate
y_pred = logreg.predict(X_test)
y_pred_proba = logreg.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy (Logistic Regression): {accuracy:.4f}")
print(classification_report(y_test, y_pred))

Accuracy (Logistic Regression): 0.6658
              precision    recall  f1-score   support

           0       0.67      0.56      0.61       902
           1       0.66      0.76      0.71      1031

    accuracy                           0.67      1933
   macro avg       0.67      0.66      0.66      1933
weighted avg       0.67      0.67      0.66      1933



In [27]:
xgb_model = XGBClassifier(n_estimators=1000, learning_rate=0.01, max_depth=5, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Predict & evaluate
y_pred = logreg.predict(X_test)
y_pred_proba = logreg.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy (XGBoost): {accuracy:.4f}")
print(classification_report(y_test, y_pred))

Accuracy (XGBoost): 0.6658
              precision    recall  f1-score   support

           0       0.67      0.56      0.61       902
           1       0.66      0.76      0.71      1031

    accuracy                           0.67      1933
   macro avg       0.67      0.66      0.66      1933
weighted avg       0.67      0.67      0.66      1933



In [28]:
models = {
    "Logistic Regression": logreg,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

# Tạo list để lưu kết quả tổng hợp
report_data = []

print(f"{'Model Name':<20} | {'Train Acc':<10} | {'Test Acc':<10} | {'Diff (Gap)':<10} | {'Status'}")
print("-" * 80)

for name, model in models.items():
    # 1. Huấn luyện
    model.fit(X_train, y_train)

    # 2. Dự đoán trên cả 2 tập
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # 3. Tính điểm
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    # 4. Tính độ chênh lệch (Gap) để bắt bệnh Overfit
    gap = train_acc - test_acc

    # 5. Logic đánh giá
    if gap > 0.05:  # Nếu Train hơn Test quá 5% -> Có dấu hiệu Overfit
        status = "Overfitting ⚠️"
    elif train_acc < 0.70: # Nếu Train thấp -> Underfit
        status = "Underfitting 📉"
    else:
        status = "Good Fit ✅"

    # In kết quả theo dạng bảng
    print(f"{name:<20} | {train_acc:.4f}     | {test_acc:.4f}     | {gap:.4f}     | {status}")



Model Name           | Train Acc  | Test Acc   | Diff (Gap) | Status
--------------------------------------------------------------------------------
Logistic Regression  | 0.6509     | 0.6658     | -0.0149     | Underfitting 📉
Random Forest        | 0.7122     | 0.6891     | 0.0232     | Good Fit ✅
XGBoost              | 0.7034     | 0.6865     | 0.0169     | Good Fit ✅
